# kernel analysis

GPU profiling and reconstruction quality analysis for turboquant-gpu.

Run `quickstart.ipynb` first to install all dependencies. This notebook assumes the environment is already set up.

## setup

Load the model and create the engine. Same setup as the quickstart.

In [ ]:
import torch, time
from turboquant_gpu import TurboQuantEngine
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    gpu = torch.cuda.get_device_name()
    sm  = torch.cuda.get_device_capability()
    print(f"{gpu}  |  sm_{sm[0]}{sm[1]}  |  CUDA {torch.version.cuda}")
else:
    print("no GPU, running on CPU")

model_id = "mistralai/Mistral-7B-v0.1"
tok   = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map=device)

head_dim = model.config.hidden_size // model.config.num_attention_heads
engine   = TurboQuantEngine(head_dim=head_dim, total_bits=3, device=device)
print(f"head_dim={head_dim}  |  device={device}")

## profiling

Profiles the fused KV compression and value decompression pipeline using `torch.profiler`. The fused kernel compresses both keys and values in a single launch.

NVTX ranges are added around each stage so they show up as labeled regions in Nsight. The cell also exports a chrome trace you can open in `chrome://tracing`.

For deeper GPU analysis, run this notebook under Nsight from terminal:

```bash
# timeline view (nsight systems)
nsys profile --trace=cuda,nvtx -o turboquant python -m jupyter nbconvert --execute kernel_analysis.ipynb

# per-kernel occupancy, memory throughput, warp stalls (nsight compute)
ncu --nvtx --nvtx-include "turboquant/" --set full -o turboquant_ncu python -m jupyter nbconvert --execute kernel_analysis.ipynb
```

In [ ]:
if device == "cuda":
    from torch.profiler import profile, ProfilerActivity

    inputs = tok("The University of Waterloo is known for ", return_tensors="pt").to(device)
    with torch.no_grad():
        fwd = model(**inputs, use_cache=True)

    kv_keys, kv_vals = engine._extract_kv(fwd.past_key_values)
    K0 = kv_keys[0][0, 0].half().contiguous()
    V0 = kv_vals[0][0, 0].half().contiguous()

    for _ in range(5):
        ck, cv = engine._compress_kv_fused(K0, V0)
        _ = engine._decompress_values(cv)
    torch.cuda.synchronize()

    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=True,
        with_stack=False,
    ) as prof:
        for _ in range(20):
            torch.cuda.nvtx.range_push("turboquant/compress_kv")
            ck, cv = engine._compress_kv_fused(K0, V0)
            torch.cuda.nvtx.range_pop()

            torch.cuda.nvtx.range_push("turboquant/decompress_v")
            dv = engine._decompress_values(cv)
            torch.cuda.nvtx.range_pop()
        torch.cuda.synchronize()

    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=20))

    prof.export_chrome_trace("turboquant_trace.json")
    print("\ntrace saved to turboquant_trace.json (open in chrome://tracing)")
else:
    print("skipping profiler (no GPU)")

## quality comparison: turboquant vs mxfp4 vs nvfp4

Compresses layer 0 of the real KV cache with three different methods and compares reconstruction quality (cosine similarity, MSE) and compression ratio.

**TurboQuant** uses Lloyd-Max quantization against N(0, 1/d) after random rotation, giving 3 bits per element. **MXFP4** uses E2M1 FP4 values with a shared E8M0 scale per group of 32. **NVFP4** uses the same FP4 values but with a finer FP8 E4M3 scale per group of 16, which gives better quality at the cost of slightly lower compression.

In [ ]:
import torch.nn.functional as F

FP4_VALUES = torch.tensor([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0])

def _quantize_fp4(x_flat, scales):
    x_scaled = x_flat / scales
    signs = x_scaled.sign()
    mags = x_scaled.abs()
    fp4 = FP4_VALUES.to(x_flat.device)
    idx = (mags.unsqueeze(-1) - fp4).abs().argmin(dim=-1)
    return fp4[idx] * signs

def mxfp4_compress(x, group_size=32):
    shape = x.shape
    x_flat = x.float().reshape(-1, group_size)
    scales = x_flat.abs().amax(dim=-1, keepdim=True) / 6.0
    scales = torch.clamp(scales, min=1e-8)
    x_q = _quantize_fp4(x_flat, scales)
    return x_q, scales, shape

def nvfp4_compress(x, group_size=16):
    shape = x.shape
    x_flat = x.float().reshape(-1, group_size)
    scales = x_flat.abs().amax(dim=-1, keepdim=True) / 6.0
    scales = torch.clamp(scales, min=1e-8)
    x_q = _quantize_fp4(x_flat, scales)
    return x_q, scales, shape

def fp4_decompress(x_q, scales, shape):
    return (x_q * scales).reshape(shape).half()

inputs = tok("The University of Waterloo is known for ", return_tensors="pt").to(device)
with torch.no_grad():
    fwd = model(**inputs, use_cache=True)

kv_keys, kv_vals = engine._extract_kv(fwd.past_key_values)
K_orig = kv_keys[0][0].half()
V_orig = kv_vals[0][0].half()

tq_ck = [engine._compress_keys(K_orig[h].contiguous()) for h in range(K_orig.shape[0])]
tq_cv = [engine._compress_values(V_orig[h].contiguous()) for h in range(V_orig.shape[0])]
K_tq = torch.stack([ck["k_mse"] for ck in tq_ck])
V_tq = torch.stack([engine._decompress_values(cv) for cv in tq_cv])

K_mxq, K_mxs, K_mxsh = mxfp4_compress(K_orig, group_size=32)
V_mxq, V_mxs, V_mxsh = mxfp4_compress(V_orig, group_size=32)
K_mxfp4 = fp4_decompress(K_mxq, K_mxs, K_mxsh)
V_mxfp4 = fp4_decompress(V_mxq, V_mxs, V_mxsh)

K_nvq, K_nvs, K_nvsh = nvfp4_compress(K_orig, group_size=16)
V_nvq, V_nvs, V_nvsh = nvfp4_compress(V_orig, group_size=16)
K_nvfp4 = fp4_decompress(K_nvq, K_nvs, K_nvsh)
V_nvfp4 = fp4_decompress(V_nvq, V_nvs, V_nvsh)

def metrics(orig, recon, label):
    cos = F.cosine_similarity(orig.float().flatten(), recon.float().flatten(), dim=0).item()
    mse = F.mse_loss(recon.float(), orig.float()).item()
    print(f"  {label:<22s}  cosine={cos:.4f}   mse={mse:.6f}")

print(f"layer 0  |  {K_orig.shape[0]} heads  |  seq_len={K_orig.shape[1]}\n")
print("keys:")
metrics(K_orig, K_tq,     "turboquant 3-bit")
metrics(K_orig, K_mxfp4,  "mxfp4 (group=32)")
metrics(K_orig, K_nvfp4,  "nvfp4 (group=16)")
print("\nvalues:")
metrics(V_orig, V_tq,     "turboquant 3-bit")
metrics(V_orig, V_mxfp4,  "mxfp4 (group=32)")
metrics(V_orig, V_nvfp4,  "nvfp4 (group=16)")

n_elem = K_orig.numel() + V_orig.numel()
fp16_bytes = K_orig.nbytes + V_orig.nbytes

tq_bits = 3 * n_elem
tq_norms = K_orig.shape[0] * K_orig.shape[1] * 2 * 2
tq_bytes = tq_bits / 8 + tq_norms

mxfp4_data = n_elem * 0.5
mxfp4_scales = (n_elem / 32) * 1
mxfp4_total = mxfp4_data + mxfp4_scales

nvfp4_data = n_elem * 0.5
nvfp4_scales = (n_elem / 16) * 1
nvfp4_total = nvfp4_data + nvfp4_scales

print(f"\ncompression:")
print(f"  fp16 baseline         {fp16_bytes/1024:.1f} KB")
print(f"  turboquant 3-bit      {tq_bytes/1024:.1f} KB   ({fp16_bytes/tq_bytes:.2f}x)")
print(f"  mxfp4 (group=32)     {mxfp4_total/1024:.1f} KB   ({fp16_bytes/mxfp4_total:.2f}x)")
print(f"  nvfp4 (group=16)     {nvfp4_total/1024:.1f} KB   ({fp16_bytes/nvfp4_total:.2f}x)")